In [0]:
import requests
import json

In [0]:
webhook_url = dbutils.secrets.get(
    scope="prj-secret",
    key="slack-webhook"
)

print("Secret Loaded Successfully")

In [0]:
def send_slack_message(message):

    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "text": message
    }

    response = requests.post(
        webhook_url,
        headers=headers,
        data=json.dumps(payload)
    )

    print(response.status_code)

In [0]:
send_slack_message("""
🚀 Databricks Connected Successfully

Secret Scope : Working

Slack Alert : Test Successful
""")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Step 1.3 - Set Catalog & Schema
spark.sql("USE CATALOG Bronze_Catalog")
spark.sql("USE SCHEMA Bronze_SCH")

In [0]:
# Step 1.4 - Read Parquet from ADLS

# Replace <storage-account> with your storage account name.

energy_df = spark.read.parquet(
    "abfss://rev1@adlsstgacntp2026.dfs.core.windows.net/Parquet_data/energy_usage_stream_v2.parquet"
)

# Step 1.5 - Verify Data
display(energy_df)

In [0]:
# Step 1.6 - Verify Schema
energy_df.printSchema()

In [0]:
# Step 1.7 - Check Record Count
print(energy_df.count())

In [0]:
# Convert Timestamp
from pyspark.sql.functions import to_timestamp,col

energy_df=energy_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"),"dd-MM-yyyy HH:mm")
)

In [0]:
last_watermark=spark.sql("""

SELECT last_processed_timestamp

FROM Bronze_Catalog.Bronze_SCH.Watermark_Table

WHERE table_name='Bronze_Energy_Usage'

""").collect()[0][0]

print(last_watermark)

In [0]:
# Incremental Filter

incremental_df=energy_df.filter(
col("timestamp")>last_watermark
)

In [0]:
if incremental_df.count() == 0:
    print("No new records found.")
else:
    print(f"New Records Found : {incremental_df.count()}")

In [0]:
print(incremental_df.count())

display(incremental_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = incremental_df.count()

try:

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("Bronze_Catalog.Bronze_SCH.Bronze_Energy_Usage")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Energy_Usage',
            'Energy Bronze Pipeline',
            current_timestamp(),
            {records_loaded},
            'SUCCESS',
            NULL
        )
    """)

    print(f"✅ Bronze_Energy_Usage loaded successfully. Records Loaded: {records_loaded}")

    send_slack_message(f"""
✅ Bronze_Energy_Usage SUCCESS

Pipeline : Energy Bronze Pipeline

Records Loaded : {records_loaded}

Status : SUCCESS
""")

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Energy_Usage',
            'Energy Bronze Pipeline',
            current_timestamp(),
            0,
            'FAILED',
            '{error_message}'
        )
    """)

    print(f"❌ Bronze_Energy_Usage failed: {error_message}")

    send_slack_message(f"""
❌ Bronze_Energy_Usage FAILED

Pipeline : Energy Bronze Pipeline

Error :

{error_message}
""")

    raise

In [0]:
spark.sql("""
UPDATE Bronze_Catalog.Bronze_SCH.Watermark_Table
SET last_processed_timestamp = (
    SELECT MAX(timestamp)
    FROM Bronze_Catalog.Bronze_SCH.Bronze_Energy_Usage
)
WHERE table_name = 'Bronze_Energy_Usage'
""")

print("✅ Watermark Updated Successfully")

In [0]:
# Verify Watermark
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name = 'Bronze_Energy_Usage'
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Audit_Log
WHERE table_name = 'Bronze_Energy_Usage'
ORDER BY load_time DESC
LIMIT 5
""").show(truncate=False)